# Verify WidowX AI robot control

Verify a WidowX AI position controller (`src/robot/wxai`): connect, read the state, go to the home pose, make small Cartesian test movements, open and close the gripper, and park the arm in the **sleep pose**.

The `driver` of the robot exprun selects the arm:

* `position_controller_wxai_00`: the real arm through `trossen_arm`, at the `ip_address` of the exprun (override it per machine in the `_sysdep` overlay). Clear the paths to home, the test poses and sleep: the reachability check does not detect collisions.
* `position_controller_fake_wxai_00`: the kinematic fake driver, for machines without the arm.
* `position_controller_mujoco_wxai_00`: the MuJoCo simulation.

The states measured along the way are saved to `verification.yaml` in the result directory of the exprun.

In [ ]:
# If it is set to discard-old, the exprun will be recreated from scratch
creation_style = "exist-ok"
# If not None, set an external experiment path
expruns_path = None
# If not None, set an output path
results_path = None

experiment = "robot_wxai"
# run = "position_controller_wxai_00"
# run = "position_controller_mujoco_wxai_00"
run = "position_controller_fake_wxai_00"

In [ ]:
import sys
sys.path.append("..")

import pathlib
from copy import copy
import yaml
from exp_run_config import Config
from robot.wxai import PositionController, WXAICommand

Config.PROJECTNAME = "BerryPicker"
if expruns_path:
    Config().set_exprun_path(expruns_path)
if results_path:
    Config().set_results_path(results_path)
exp = Config().get_experiment(experiment, run, creation_style=creation_style)
print(f"Driver: {exp['driver']}, IP address: {exp['ip_address']}")

## Enable motion deliberately

The fake driver and the MuJoCo simulation move without asking. For the real arm, set `ALLOW_ROBOT_MOTION` to True only after checking the workspace. Each blocking test movement returns to the home pose.

In [ ]:
ALLOW_ROBOT_MOTION = exp["driver"] in ("fake", "mujoco")
MOVING_TIME = 2.0
test_offsets = {
    "z": 0.02,       # 2 cm upward
    "x": 0.02,       # 2 cm forward
}

In [ ]:
if not ALLOW_ROBOT_MOTION:
    raise RuntimeError("Set ALLOW_ROBOT_MOTION = True after checking the workspace")

states = {}
rob = PositionController(exp)
rob.start_robot()
try:
    states["start"] = rob.get_state()
    print("Connected. Measured starting pose:")
    print(rob.get_position())
    rob.go_home(moving_time=MOVING_TIME)
    home = rob.get_position()
    states["home"] = rob.get_state()
    print("Home pose:")
    print(home)
    for field, offset in test_offsets.items():
        target = copy(home)
        target[field] += offset
        if not rob.can_reach(target):
            raise ValueError(f"Test target is not reachable:\n{target}")
        rob.move(WXAICommand(target), moving_time=MOVING_TIME, blocking=True)
        states[f"moved_{field}"] = rob.get_state()
        print(f"Moved {field} by {offset}, error: {rob.get_position()[field] - target[field]:.4f} m")
        rob.move(WXAICommand(home), moving_time=MOVING_TIME, blocking=True)
    rob.move(WXAICommand(home, "release"), moving_time=MOVING_TIME, blocking=True)
    rob.update(1.0)
    states["released"] = rob.get_state()
    rob.move(WXAICommand(home, "grasp"), moving_time=MOVING_TIME, blocking=True)
    rob.update(1.0)
    states["grasped"] = rob.get_state()
    print(f"Gripper released: {states['released']['gripper_position']:.4f} m, "
          f"grasped: {states['grasped']['gripper_position']:.4f} m")
finally:
    rob.stop_robot()
print("Stopped in the sleep pose")

In [ ]:
with open(pathlib.Path(exp.data_dir(), "verification.yaml"), "w") as f:
    yaml.dump(states, f)
print(f"Saved the measured states to {exp.data_dir()}")

In [ ]:
exp.done()